# Fine-tune ScamBench on your Windows NVIDIA machine

Extract `gpu_bundle.zip` into a folder on the GPU machine. Open **this extracted notebook** in VS Code and select the Python kernel with CUDA-enabled PyTorch. No Git clone, original corpus download, or Hugging Face login is required; the base model downloads on the first run.

Run cells in order. First run the two-step smoke check, then train **English-only** and **English + silver Kazakh**. Both select checkpoints on the same separate EN + silver development families. The 300 paired EN/KK examples are used only by the final evaluation cell.

These 300 are a historical benchmark used in earlier experiments, and their Kazakh has not had native-speaker review. Development Kazakh is machine translated. The new holdout reduces training data, so we rerun matched TF-IDF baselines rather than claiming direct comparability with the old 0.687/0.898 results.

Model: [FacebookAI/xlm-roberta-base](https://huggingface.co/FacebookAI/xlm-roberta-base). This trains a three-class classifier; binary metrics merge its predictions. No external logging or uploads are enabled.

In [ ]:
import sys
import pathlib
import subprocess
import json

# Run with the notebook working directory set to the extracted bundle folder.
ROOT = pathlib.Path.cwd()
assert (ROOT / "manifest.json").exists(), f"Open the extracted notebook; current folder is {ROOT}"
print("Python:", sys.executable)
print("Bundle:", ROOT)
print(json.dumps(json.loads((ROOT / "manifest.json").read_text())["counts"], indent=2))

Install the training libraries into this kernel. CUDA-enabled PyTorch should already be installed. If the CUDA check below fails, select your CUDA kernel or install the appropriate PyTorch build using the [official selector](https://pytorch.org/get-started/locally/), then restart the kernel. This cell does not choose a CUDA build for you.

In [ ]:
%pip install "transformers==4.57.6" "accelerate>=1.1,<2" "sentencepiece>=0.2,<0.3" "scikit-learn>=1.6,<1.8"

In [ ]:
import torch
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
assert torch.cuda.is_available(), "This kernel cannot use CUDA. Fix the PyTorch/kernel installation first."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
subprocess.run([sys.executable, "train_transformer.py", "--help"], cwd=ROOT, check=True);

Set the output location on a disk with ample free space (allow roughly 15 GB for model downloads, optimizer checkpoints and two trained models). Results persist here when the kernel stops. Start with batch size 2; if CUDA runs out of memory, use 1 and accumulation 32. The effective batch size is 32 on a single GPU.

Default: 3 epochs, learning rate 2e-5, seed 42, at most 512 tokens. Long inputs retain their beginning and end; the runner records how many are shortened. This is a first controlled run, not a hyperparameter sweep.

In [ ]:
OUTPUT = ROOT / "gpu_results"
OUTPUT.mkdir(exist_ok=True)
BATCH_SIZE = 2
ACCUMULATION = 16
SEED = 42

def run(*arguments):
    subprocess.run([sys.executable, "-u", str(ROOT / "train_transformer.py"),
                    *map(str, arguments)], cwd=ROOT, check=True)

def train_variant(variant, smoke=False):
    destination = OUTPUT / ("smoke" if smoke else f"{variant}_seed{SEED}")
    options = ["train", "--data", ROOT, "--variant", variant, "--output", destination,
               "--batch-size", BATCH_SIZE, "--accumulation", ACCUMULATION,
               "--seed", SEED]
    if smoke:
        options += ["--smoke"]
    run(*options)

train_variant("mixed", smoke=True)

The randomly initialized classification-head warning is expected. Once the smoke run saves a checkpoint and validation metrics, start the real runs. Existing model directories are protected against accidental overwrites.

For an interrupted full run, resume with the same settings and append `--resume` plus its latest `checkpoint-N` path to the runner command. A smoke checkpoint cannot be evaluated as a full experiment.

In [ ]:
train_variant("en")

In [ ]:
train_variant("mixed")

Inspect validation metrics before touching the historical test set. Both variants use exactly the same development data. Do not choose epochs, learning rates or token length by repeatedly looking at the test results.

In [ ]:
for variant in ["en", "mixed"]:
    folder = OUTPUT / f"{variant}_seed{SEED}"
    print(variant, json.loads((folder / "validation_metrics.json").read_text()))
    config = json.loads((folder / "run.json").read_text())
    print("Shortened train/dev inputs:", config["train_truncated"], config["dev_truncated"])

## Final evaluation

Run this after both training runs finish and settings are fixed. It evaluates the saved best validation checkpoint on paired EN-300 and KK-300, then fits four TF-IDF baselines on the **same reduced training pool**. No checkpoint selection uses test scores.

For stronger evidence, repeat the fixed training settings with seeds 0 and 1 and report all seeds; use a new reviewed test set for future model development. A first single-seed result is exploratory.

In [ ]:
for variant in ["en", "mixed"]:
    run("evaluate", "--data", ROOT, "--output", OUTPUT / f"{variant}_seed{SEED}",
        "--batch-size", BATCH_SIZE)
run("baselines", "--data", ROOT, "--output", OUTPUT / "baselines")

In [ ]:
print(f'{"model":<24} {"EN F1":>9} {"KK F1":>9} {"KK binary":>11}')
def show(name, metrics):
    print(f'{name:<24} {metrics["test_en"]["macro_f1"]:>9.4f} '
          f'{metrics["test_kk"]["macro_f1"]:>9.4f} {metrics["test_kk"]["binary_macro_f1"]:>11.4f}')

for variant in ["en", "mixed"]:
    show(f"XLM-R {variant}", json.loads((OUTPUT / f"{variant}_seed{SEED}" / "test_metrics.json").read_text()))
for name, metrics in json.loads((OUTPUT / "baselines" / "baselines.json").read_text()).items():
    show(name, metrics)

Zip the small result files to bring back for analysis. This excludes large model weights and checkpoints; keep those on your GPU machine. The archive includes predictions, run settings, package versions, training logs and baseline metrics.

In [ ]:
import zipfile
archive = ROOT / "gpu_results_summary.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(ROOT / "manifest.json", "manifest.json")
    for path in OUTPUT.rglob("*.json"):
        if not any(part.startswith("checkpoint-") for part in path.relative_to(OUTPUT).parts):
            if path.name not in {"tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "config.json"}:
                z.write(path, path.relative_to(ROOT))
print("Bring this file back:", archive)